In [10]:
from bz2 import BZ2Decompressor
url = 'https://dumps.wikimedia.org/enwikivoyage/latest/enwikivoyage-latest-pages-articles.xml.bz2'
decompressed = ('.').join(url.split('/')[-1].split('.')[:-1])
if not os.path.isfile(decompressed):
    with open(decompressed, 'wb') as new_file, open(url.split('/')[-1], 'rb') as file:
        decompressor = BZ2Decompressor()
        for data in iter(lambda : file.read(100 * 1024), b''):
            new_file.write(decompressor.decompress(data))


In [15]:
if not os.path.isfile('wikivoyage_latest_articles_text.json'):
    import sys
    !{sys.executable} -m pip install xmltodict

    import xmltodict

    #this step takes some time
    with open('enwikivoyage-latest-pages-articles.xml') as fd:
        doc = xmltodict.parse(fd.read())

    data = doc['mediawiki']['page']
    print('To process %s records' %len(data))
    del doc

    from collections import defaultdict
    completed = 0
    articles = defaultdict(list)
    
    for item in data:
        if 'redirect' not in item:
            try:
                articles[item['title']].append(item['revision']['text']['#text'])
            except KeyError:
                continue

        completed+=1
        if completed%10000==0 or completed==len(data):
            print('Completed %s' %completed)

    print('Found %s articles' %len(articles))
    for article, text in articles.items():
        articles[article] = "".join(text)

    with open('wikivoyage_latest_articles_text.json', 'w') as f:
        json.dump(articles, f)

    del articles
    del data

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
To process 75115 records
Completed 10000
Completed 20000
Completed 30000
Completed 40000
Completed 50000
Completed 60000
Completed 70000
Found 41971 articles


In [16]:
import unicodedata
import re
with open('wikivoyage_latest_articles_text.json', 'r') as f:
    consolidated = json.load(f)

print(len(consolidated))

cleaned = {}
completed = 0
issues = 0

for article_name in consolidated:
#   1. ignore articles which are not destinations (from article name and article tags)
    if not article_name.startswith('Module') and not article_name.startswith('Template:') and not article_name.startswith('Category:')\
    and not article_name.startswith('File:') and not article_name.startswith('Wikivoyage:') and not article_name.startswith('MediaWiki:') and not article_name in ['Moon', 'Space']\
        and len(re.findall('{{outlinetopic}}', consolidated[article_name].lower()))==0 and len(re.findall('{{usabletopic}}', consolidated[article_name].lower()))==0\
        and len(re.findall('{{guidetopic}}', consolidated[article_name].lower())) == 0 and len(re.findall('{{startopic}}', consolidated[article_name].lower()))==0\
        and len(re.findall('{{disamb}}', consolidated[article_name].lower()))==0 and len(re.findall('{{disambig}}', consolidated[article_name].lower()))==0 and len(re.findall('{{disambiguation}}', consolidated[article_name].lower()))==0\
        and len(re.findall('{{itinerary}}', consolidated[article_name].lower()))==0 \
        and len(re.findall('{{usablephrasebook}}', consolidated[article_name].lower()))==0 and len(re.findall('{{phrasebookguide}}', consolidated[article_name].lower()))==0 \
        and len(re.findall('{{Title-Index page}}', consolidated[article_name]))==0 \
        and len(re.findall('{{GalleryPageOf.*}}', consolidated[article_name]))==0 \
        and len(re.findall('{{stub}}', consolidated[article_name].lower())) == 0 \
        and len(re.findall('{{historical}}', consolidated[article_name].lower())) == 0:

        # 2. get 'ispartof' tags
        IsPartOf = re.findall('{{IsPartOf.*}}', consolidated[article_name]) + re.findall('{{isPartOf.*}}', consolidated[article_name])

        # 3. get geo tags
        geo = re.findall('{{geo.*}}', consolidated[article_name].lower())

        # 4. get page type tags
        city = re.findall('{{usablecity}}', consolidated[article_name].lower()) + re.findall('{{outlinecity}}', consolidated[article_name].lower()) \
                    + re.findall('{{guidecity}}', consolidated[article_name].lower()) + re.findall('{{starcity}}', consolidated[article_name].lower()) \
                    + re.findall('{{ussblecity}}', consolidated[article_name].lower())

        country = re.findall('{{usablecountry}}', consolidated[article_name].lower()) + re.findall('{{outlinecountry}}', consolidated[article_name].lower()) \
                    + re.findall('{{guidecountry}}', consolidated[article_name].lower()) + re.findall('{{starcountry}}', consolidated[article_name].lower())

        district = re.findall('{{usabledistrict}}', consolidated[article_name].lower()) + re.findall('{{outlinedistrict}}', consolidated[article_name].lower()) \
                    + re.findall('{{guidedistrict}}', consolidated[article_name].lower())+ re.findall('{{stardistrict}}', consolidated[article_name].lower())

        region = re.findall('{{usableregion}}', consolidated[article_name].lower()) + re.findall('{{outlineregion}}', consolidated[article_name].lower()) \
                    + re.findall('{{guideregion}}', consolidated[article_name].lower()) + re.findall('{{extraregion\|subregion=yes}}', consolidated[article_name].lower()) \
                    + re.findall('{{starregion}}', consolidated[article_name].lower()) + re.findall('{{extraregion\|subregion=no}}', consolidated[article_name].lower()) \
                    + re.findall('{{extraregion}}', consolidated[article_name].lower())

        airport = re.findall('{{usableairport}}', consolidated[article_name].lower()) + re.findall('{{outlineairport}}', consolidated[article_name].lower())\
                    + re.findall('{{guideairport}}', consolidated[article_name].lower())

        park = re.findall('{{usablepark}}', consolidated[article_name].lower()) + re.findall('{{outlinepark}}', consolidated[article_name].lower()) \
                    + re.findall('{{guidepark}}', consolidated[article_name].lower()) + re.findall('{{starpark}}', consolidated[article_name].lower())

        diveguide = re.findall('{{usablediveguide}}', consolidated[article_name].lower()) + re.findall('{{outlinediveguide}}', consolidated[article_name].lower()) \
                    + re.findall('{{guidediveguide}}', consolidated[article_name].lower()) + re.findall('{{stardiveguide}}', consolidated[article_name].lower())

        continent = re.findall('{{usablecontinent}}', consolidated[article_name].lower()) + re.findall('{{outlinecontinent}}', consolidated[article_name].lower())

        
        # 5. clean naming before saving
        if len(geo)>0 and len(diveguide)==0 and article_name not in ['Commonwealth of Independent States']: #skip dive guides
            article_name = article_name.replace('_', ' ').split('{{')[0].strip().lower()
            
            if unicodedata.normalize('NFKD', article_name).encode('ascii', 'ignore') == 'brac':
                article_name = 'brac'
            elif unicodedata.normalize('NFKD', article_name).encode('ascii', 'ignore') == 'rugen':
                article_name = 'rugen'
            
            cleaned[article_name] = {}

            # get lat long
            if len(geo)>0:
                cleaned[article_name]['latitude'] = geo[-1].split('|')[1]
                cleaned[article_name]['longitude'] = geo[-1].split('|')[2]

            # get parents
            cleaned[article_name]['ispartof'] = []
            for parts in IsPartOf:
                parent = parts.split('|')[1].replace('}','').replace('_', ' ').split('{{')[0].strip().lower()

                
                #fixes for inconsistent data
                if parent == 'ko pha ngan':
                    parent = 'ko pha-ngan'
                elif parent in ['lowland shandong', 'highland shandong', 'coastal shandong']:
                    parent = 'shandong'
                elif parent in ['southern delaware', 'northern delaware', 'central delaware']:
                    parent = 'delaware'
                elif parent in ['burgraviate', 'puster valley', 'eisack valley']:
                    parent = 'south tyrol'
                elif parent == 'bohemian-moravian highlands':
                    parent = 'highlands (czech republic)'
                elif parent == 'brahmanbaria district':
                    parent = 'chittagong division'
                elif parent == 'eastern desert':
                    parent = 'eastern desert (jordan)'
                elif parent == 'caribbean coast':
                    parent = 'caribbean coast (guatemala)'
                elif parent == 'santander (colombia)':
                    parent = 'santander (department, colombia)'
                elif parent == 'tripolitania':
                    parent = 'libya'
                elif parent == 'wooster area ohio':
                    parent = 'wooster area'
                elif parent == 'tatra mountains (poland)':
                    parent = 'tatra national park (poland)'
                elif parent == 'salcette':
                    parent = 'salcete'
                elif parent == 'eastern barbados':
                    parent = 'central eastern barbados'
                elif parent == 'east khasi hills':
                    parent = 'meghalaya'
                elif parent == 'samar':
                    parent = 'samar (philippines)'
                elif parent == 'chikmagalur (district)' and article_name != 'chikmagalur' :
                    parent = 'chikmagalur'
                elif unicodedata.normalize('NFKD', parent).encode('ascii', 'ignore') == 'rugen':
                    parent = 'rugen'
                elif article_name == 'chikmagalur':
                    parent = 'karnataka'
        
                cleaned[article_name]['ispartof'].append(parent)
                
                
            # 7. get destination type
            if len(airport)>0:
                cleaned[article_name]['type']='airport'
            elif len(city)>0:
                cleaned[article_name]['type']='city'
            elif len(continent)>0:
                cleaned[article_name]['type']='continent'
            elif len(country)>0:
                cleaned[article_name]['type']='country'
            elif len(district)>0:
                cleaned[article_name]['type']='district'
            elif len(park)>0:
                cleaned[article_name]['type']='park'
            elif len(region)>0:
                cleaned[article_name]['type']='region'



    completed +=1
    if completed%1000==0 or completed==len(consolidated):
        print('Completed: %s' %completed)


print('Total sorted: %s' %len(cleaned))

with open('destination_details.json', 'w') as f:
    json.dump(cleaned, f)

del consolidated

41971
Completed: 1000
Completed: 2000
Completed: 3000
Completed: 4000
Completed: 5000
Completed: 6000
Completed: 7000
Completed: 8000
Completed: 9000
Completed: 10000
Completed: 11000
Completed: 12000
Completed: 13000
Completed: 14000
Completed: 15000
Completed: 16000
Completed: 17000
Completed: 18000
Completed: 19000
Completed: 20000
Completed: 21000
Completed: 22000
Completed: 23000
Completed: 24000
Completed: 25000
Completed: 26000
Completed: 27000
Completed: 28000
Completed: 29000
Completed: 30000
Completed: 31000
Completed: 32000
Completed: 33000
Completed: 34000
Completed: 35000
Completed: 36000
Completed: 37000
Completed: 38000
Completed: 39000
Completed: 40000
Completed: 41000
Completed: 41971
Total sorted: 28040


In [19]:
with open('destination_details.json', 'r') as f:
    cleaned = json.load(f)


def map_destinations(mapped, current_dict):
    global destination_mapping, parent, destination
    if not mapped:
        if parent in current_dict:
            current_dict[parent].update({destination: {}})
            mapped = True
            
            #find if any of the top level keys match this article
            if destination in destination_mapping:
                current_dict[parent][destination] = destination_mapping.pop(destination)
        
        # if parent not in dict but article in dict. given previous step, this can only happen if destination is at top level
        elif destination in current_dict and current_dict==destination_mapping:
            current_dict[parent] = {}
            current_dict[parent][destination] = current_dict.pop(destination)
            mapped = True
            attached = False
            #find if any of the values match this parent
            step_through_dict(attached, destination_mapping)

        else:
            for next_level in current_dict:
                mapped, current_dict[next_level] = map_destinations(mapped, current_dict[next_level])
    
    return mapped, current_dict


def step_through_dict(attached, current_dict):
    global destination_mapping, destination, parent

    iter_list = list(current_dict.keys())
    for item in iter_list:
        if not attached:
            if item == parent and current_dict!=destination_mapping:
                current_dict[parent][destination] = destination_mapping.pop(parent)[destination]
                attached = True
            elif len(current_dict[item])>0:
                attached = step_through_dict(attached, current_dict[item])
    return attached
        
    
destination_mapping = {}
print('To process %s records' %len(cleaned))
processed = 0
for destination in cleaned:
    for parent in cleaned[destination]['ispartof']:
            
        mapped = False
        mapped, destination_mapping = map_destinations(mapped, destination_mapping)
        
        if not mapped:
            destination_mapping[parent] = {}
            destination_mapping[parent][destination] = {}
    processed+=1
    if processed%1000==0 or processed==len(cleaned):
        print('Completed: %s' %processed)

with open('destination_mapping.json', 'w') as f:
    json.dump(destination_mapping, f)
    
del destination_mapping


To process 28040 records
Completed: 1000
Completed: 2000
Completed: 3000
Completed: 4000
Completed: 5000
Completed: 6000
Completed: 7000
Completed: 8000
Completed: 9000
Completed: 10000
Completed: 11000
Completed: 12000
Completed: 13000
Completed: 14000
Completed: 15000
Completed: 16000
Completed: 17000
Completed: 18000
Completed: 19000
Completed: 20000
Completed: 21000
Completed: 22000
Completed: 23000
Completed: 24000
Completed: 25000
Completed: 26000
Completed: 27000
Completed: 28000
Completed: 28040


In [49]:
import json

with open('destination_details.json', 'r') as f:
    details = json.load(f)

def get_parent(current, chain=''):
    if chain == '':
        chain=current.lower()
        current=current.lower()
    try:
        for parent in details[current]['ispartof']:
            chain = '%s > %s' %(parent, chain)
            chain = get_parent(parent, chain)
    except KeyError:
        return chain
    else:
        return chain

def get_child(search):
    child_articles = []
    for article in details:
        for parent in details[article]['ispartof']:
            if parent == search.lower():
                child_articles.append((article,details[article]))
    return child_articles

destination = 'luxembourg'
print(get_parent(destination))
for item in get_child(destination):
    print(item)
    
del details

europe > benelux > luxembourg
('mullerthal', {'latitude': '49.795', 'longitude': '6.406', 'ispartof': ['luxembourg'], 'type': 'region'})
('land of the red rocks', {'latitude': '49.510', 'longitude': '5.993}}', 'ispartof': ['luxembourg']})
('moselle valley (luxembourg)', {'latitude': '49.639', 'longitude': '6.389', 'ispartof': ['luxembourg'], 'type': 'region'})
('éislek', {'latitude': '49.959', 'longitude': '6.025', 'ispartof': ['luxembourg'], 'type': 'region'})
('central luxembourg', {'latitude': '49.583333', 'longitude': '6.166667', 'ispartof': ['luxembourg'], 'type': 'region'})


In [93]:
import requests
from bs4 import BeautifulSoup

fr_base_url = "https://flight-report.com"
sr_url = "https://flight-report.com/en/search/result?classes=1&classes=4&classes=2&classes=3&classes=5&year_min=2024&year_max=2024"

data = requests.get(sr_url)
soup = BeautifulSoup(data.content, 'html.parser')
# Find all search result 'div' elements
result_divs = soup.select("div#reports div.box")
list_of_links = soup.select("div#reports div.box article.thumb a")

# Extract links directly
list_of_links = [link['href'] for link in list_of_links if 'member' not in link['href']]

def process(data):
    cleaned = {}
    soup = BeautifulSoup(data.content, 'html.parser')
    # Extract flight title (if structured as expected)
    title_text = soup.select_one('h1').text.strip()
    flight_code, route = title_text.split('Review of ')[1].split(' flight ')
    route, t_class = route.split(' in ')
    cleaned['flight'] = flight_code
    cleaned['starting_dest'] = route.split(' → ')[0]
    cleaned['ending_dest'] = route.split(' → ')[1]
    cleaned['class'] = t_class
    # Extract data from 'information' divs
    for info_div in soup.select('div.information'):
        key = info_div.find('span', recursive=False).text.strip()  
        value = info_div.find_all('span')[-1].text.strip() 
        cleaned[key.lower()] = value 

    airline_section = soup.find('section', id='AirlineReviews')  

    if airline_section:  # Check if the section exists
        airline_image = airline_section.find('img')
        cleaned['airline_image_url'] = airline_image.get('src')  
        cleaned['airline'] = airline_image.get('alt')  

        # cleaned['ranking'] = airline_section.find('span', class_='ranking').text.strip()

        star_icons = airline_section.find_all('i')
        star_rating = sum(
            1 if 'star-1' in icon.get('class', []) else 0.5 
            if 'star-half-alt' in icon.get('class', []) else 0  
            for icon in star_icons
        )
        cleaned['star_rating'] = star_rating

        cleaned['review_count'] = airline_section.find('a', class_='ReviewCount').text.strip()

    report_text_element = soup.find_all(id='reportText')[0]  # Using find_all

    if report_text_element:
        cleaned['report_text'] = report_text_element.text.strip().replace('\n', '').replace('\t', '')
    else:
        cleaned['report_text'] = "Report Text Not Found"
    return cleaned

for link in list_of_links[:2]:
    report = fr_base_url + link
    data = requests.get(report)
    print(process(data))


{'flight': 'BA250', 'starting_dest': 'Santiago', 'ending_dest': 'London', 'class': 'First', 'airline': 'BA', 'seat': '2K', 'aircraft': 'Boeing 787-9', 'flight time': '14:15', 'take-off': '12 Mar 24, 13:00', 'arrival at': '13 Mar 24, 06:15', 'airline_image_url': 'https://static.flight-report.com/media/compagnie/ba.jpg', 'star_rating': 0, 'review_count': '977 reviews', 'report_text': 'By bldavidGOLD15                            Published on 22nd March 2024                                                    Welcome to my flight report about British Airways\' longest flight in first class which is also the longest flight between Europe and the Americas.I took this flight to go back to Europe after my trip to Argentina, Uruguay and Chile. But why in first class? Because it cost less than half than business class on the same flight. Error fare? Business class was full and first class was empty? I don\'t know. It was also cheaper than business class from Santiago to Paris with Air France or t

In [94]:
import requests
from bs4 import BeautifulSoup

fr_base_url = "https://flight-report.com"
sr_url = "https://flight-report.com/en/search/result?classes=1&classes=4&classes=2&classes=3&classes=5&year_min=2024&year_max=2024"

data = requests.get(sr_url)
soup = BeautifulSoup(data.content, 'html.parser')
# Find all search result 'div' elements
result_divs = soup.select("div#reports div.box")
list_of_links = soup.select("div#reports div.box article.thumb a")

In [95]:
list_of_links

[<a class="avatar" href="/en/member/7228" title="See FlyAlex 's Profile">
 <img alt="FlyAlex" class="lazy gold-member" data-src="https://static.flight-report.com/media/avatars/7228/100_HCSXBPKM_7228.jpg" height="36" src="data:image/png;base64,R0lGODlhAQABAAD/ACwAAAAAAQABAAACADs=" srcset="https://static.flight-report.com/media/avatars/7228/100_HCSXBPKM_7228.jpg 2x" title="FlyAlex avatar" width="36"/>
 </a>,
 <a href="/en/report/66834/lufthansa-lh1835-gran-canaria-island-lpa-munich-muc/" title="Flight LH1835 review by FlyAlex"><span>Gran Canaria Island ✈ Munich</span></a>,
 <a class="avatar" href="/en/member/7228" title="See FlyAlex 's Profile">
 <img alt="FlyAlex" class="lazy gold-member" data-src="https://static.flight-report.com/media/avatars/7228/100_HCSXBPKM_7228.jpg" height="36" src="data:image/png;base64,R0lGODlhAQABAAD/ACwAAAAAAQABAAACADs=" srcset="https://static.flight-report.com/media/avatars/7228/100_HCSXBPKM_7228.jpg 2x" title="FlyAlex avatar" width="36"/>
 </a>,
 <a href="/e

In [150]:
import praw
from trafilatura import fetch_url, extract

from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModel
from transformers import pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained("yanekyuk/bert-uncased-keyword-extractor")
model = AutoModelForTokenClassification.from_pretrained(
    "yanekyuk/bert-uncased-keyword-extractor"
)

nlp = pipeline("ner", model=model, tokenizer=tokenizer)

def extract_keywords(text):
    """
    Extract keywords and construct them back from tokens
    """
    result = list()
    keyword = ""
    for token in nlp(text):
        if token["entity"] == "I-KEY":
            keyword += (
                token["word"][2:]
                if token["word"].startswith("##")
                else f" {token['word']}"
            )
        else:
            if keyword:
                result.append(keyword)
            keyword = token["word"]
    # Add the last keyword
    result.append(keyword)
    return list(set(result))

reddit = praw.Reddit(
    client_id="c9MNBbq8GTuZD9vVOLkC7Q",
    client_secret="yYuIt0aDZe2OSB54Z0zzBey-x3ulLQ",
    user_agent="Mozilla/5.0 (X11; CrOS x86_64 15633.69.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.6045.212 Safari/537.36",
)
def redditScraper(subReddit, amountOfPosts=10, topOfWhat='week'):
    listOfPosts = []
    for submission in reddit.subreddit(subReddit).top(topOfWhat, limit=amountOfPosts):
        post_obj = {}
        post_obj["url"] = submission.url
        post_obj["id"] = submission.id
        post_obj["author"] = submission.author.name
        post_obj["domain"] = submission.domain
        post_obj["created"] = submission.created
        post_obj["created_utc"] = submission.created_utc
        post_obj["title"] = submission.title
        post_obj["selftext"] = submission.selftext
        downloaded = fetch_url(submission.url)
        post_obj["content"] = extract(downloaded)
        post_obj["comments"] = [(comment.author.name if comment.author else "unknown", comment.body) for comment in submission.comments]
        listOfPosts.append(post_obj)

    print ("Grabing " + str(len(listOfPosts)) + " posts from r/" + subReddit)
    print ("")
    return listOfPosts

def insert_reddit_data(data: list) -> None:
    # Calculate embedding values for posts and comments
    for post in data:
        post_text = post["title"] + "\n" + post["selftext"]

        # print(post_text)

insert_reddit_data(redditScraper('news'))

/var/folders/5h/_j36d2tx7kzcvzkdgn6_g3jr0000gn/T/ipykernel_24156/1578891359.py:43: DeprecationWarning: Positional arguments for 'BaseListingMixin.top' will no longer be supported in PRAW 8.
Call this function with 'time_filter' as a keyword argument.
  for submission in reddit.subreddit(subReddit).top(topOfWhat, limit=amountOfPosts):


Grabing 10 posts from r/news



In [190]:
submission = list(reddit.subreddit('news').controversial('day', limit=6))[4]
submission.comments.replace_more(limit=0)
# for comment in submission.comments.list():
#     upvote_ratio = comment.ups / (comment.ups + comment.downs) * 100
#     print(f"Comment Body: {comment.body}")
#     print(f"Upvote Ratio: {upvote_ratio:.2f}%")
#     print(f"Ups: {comment.ups}  Down : {comment.downs}")

[{'author':comment.author.name if comment.author else "unknown", 
  'text': comment.body, 
  'score': comment.ups} for comment in submission.comments if comment.ups > 100
]
                                  

/var/folders/5h/_j36d2tx7kzcvzkdgn6_g3jr0000gn/T/ipykernel_24156/3413575123.py:1: DeprecationWarning: Positional arguments for 'BaseListingMixin.controversial' will no longer be supported in PRAW 8.
Call this function with 'time_filter' as a keyword argument.
  submission = list(reddit.subreddit('news').controversial('day', limit=6))[4]


[{'author': 'didsomebodysaymyname',
  'text': ">The City says their insurance provider will pay the $500,000 to Hall as a gross settlement, which includes court costs, attorneys fees, and expenses. To be clear, that is not taxpayer money.\n\n\nFuck off, that's using semantics to obscure the fact the insurance premium is paid with tax payer money.\n\n\nAnd this will likely result in higher premiums.",
  'score': 157}]

In [192]:
1+10

11